In [1]:
import pygsheets # use 'pip install pygsheets', not maintained with conda

import pandas
import dateti me

# terminals

## clean up terminals

In [2]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1tcS6Wd-Wp-LTDpLzFgJY_RSNDnbyubW3J_9HKIAys4A')
#spreadsheet = gc.open_by_key('1d0kLE0WmAn9b4XdugffiEaAHGWy6EhyF7zY1DM12zCc') # for Mar 2023 EGT/GGIT release

terms_df_orig = spreadsheet.worksheet('title', 'Terminals').get_as_df(start='A3')
terms_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
terms_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()
terms_copyright_df = spreadsheet.worksheet('title', 'Copyright').get_as_df()

/Users/baird/mambaforge/envs/gem/lib/python3.10/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


In [3]:
# remove oil export terminals
terms_df_orig = terms_df_orig[terms_df_orig['Fuel']!='Oil']
# remove anything without a wiki page
terms_df_orig = terms_df_orig[terms_df_orig['Wiki']!='']

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [4]:
terms_dict_df_include = terms_dict_df.copy()[terms_dict_df.copy()['IncludeWithDataRelease']=='Yes']
terms_dict_df_include = terms_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
terms_dict_df_include = terms_dict_df_include[['VariableName','Definition']]

In [5]:
terms_df_subset = terms_df_orig.copy()[terms_dict_df_include['VariableName'].tolist()]

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [6]:
excel_writer = pandas.ExcelWriter('GEM-GGIT-LNG-Terminals-'+str(datetime.date.today())+'.xlsx')#, engine='xlsxwriter')

terms_df_subset.to_excel(excel_writer, sheet_name='LNG Terminals '+str(datetime.date.today()), index=False)
terms_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
terms_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
terms_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

excel_writer.close()

# pipelines

### comment out appropriate final lines to only publish oil or gas

In [37]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')
#spreadsheet = gc.open_by_key('1bfPrp0w8Ruorq08Qe4hD8M3xVJ5e00phZ6ApFivO98k') # december 2022 gas pipelines
#spreadsheet = gc.open_by_key('1PKsCoVnfnCEalDBOF0Fmny0-pg1qy86DoReNHI-97WM') # March 2023 EGT/GGIT update

gas_pipes = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A2')
gas_pipes = gas_pipes.loc[gas_pipes.Fuel=='Gas']
oil_pipes = spreadsheet.worksheet('title','Oil/NGL pipelines').get_as_df(start='A2')
oil_pipes = oil_pipes.loc[oil_pipes.Fuel.isin(['Oil','NGL'])]

#gas_pipes = gas_pipes.drop('WKTFormat', axis=1) # delete WKTFormat column
#oil_pipes = oil_pipes.drop('WKTFormat', axis=1)

#pipes_df_orig = gas_pipes.copy(); type_to_save = 'Gas'
pipes_df_orig = oil_pipes.copy(); type_to_save = 'Oil'

pipes_dict_df = spreadsheet.worksheet('title','Data dictionary').get_as_df()
pipes_acronyms_df = spreadsheet.worksheet('title','Acronyms').get_as_df()
if type_to_save=='Gas':
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GGIT').get_as_df()
elif type_to_save=='Oil':
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GOIT').get_as_df()

/Users/baird/mambaforge/envs/gem/lib/python3.10/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up pipelines

In [38]:
# pipelines cleanup
# remove all N/A status, all empty rows (gauged by those without a Wiki page)
pipes_df_orig = pipes_df_orig[pipes_df_orig['Status']!='N/A']
pipes_df_orig = pipes_df_orig[pipes_df_orig['Wiki']!='']

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [39]:
pipes_dict_df_include = pipes_dict_df.copy()[(pipes_dict_df.copy()['IncludeWithDataRelease']=='Yes')]
pipes_dict_df_include = pipes_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
pipes_dict_df_include = pipes_dict_df_include[['VariableName','Definition']]

In [40]:
pipes_df_subset = pipes_df_orig.copy()[pipes_dict_df_include['VariableName'].tolist()]

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [41]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

if type_to_save=='Oil':
    excel_writer = pandas.ExcelWriter('GEM-GOIT-'+type_to_save+'-Pipelines-'+str(datetime.date.today())+'.xlsx')#, engine='xlsxwriter')
elif type_to_save=='Gas':
    excel_writer = pandas.ExcelWriter('GEM-GGIT-'+type_to_save+'-Pipelines-'+str(datetime.date.today())+'.xlsx')#, engine='xlsxwriter')

pipes_df_subset.to_excel(excel_writer, sheet_name='Gas Pipelines '+str(datetime.date.today()), index=False)
pipes_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
pipes_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
pipes_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

#workbook = excel_writer.book
#worksheet = excel_writer.sheets['Terminals '+str(datetime.date.today())]
#format = workbook.add_format({'text_wrap': True})

excel_writer.close()